# Chapter 7 Finetuning to Follow Instructions

## 7.1 Introduction to instruction fine-tuning

# LLM Pretraining vs. Instruction Fine-Tuning: Summary

## 1. Pretraining Phase
* **Training Procedure:** The LLM is trained to generate text **one word at a time**[cite: 1].
* **Capability:** It excels at **text completion** (e.g., finishing sentences or writing paragraphs given a fragment input)[cite: 1].
* **Limitation:** Pretrained models frequently struggle with **specific instructions** (e.g., *"Fix the grammar in this text"*)[cite: 1].

---

## 2. Instruction Fine-Tuning Phase
* **Definition:** Also known as **supervised instruction fine-tuning**, which uses the pretrained LLM as its foundation[cite: 1].
* **Objective:** Focuses on improving the LLM’s ability to **follow instructions** and generate the desired response[cite: 1].

---

## 3. The Fine-Tuning Process
The workflow consists of three major stages, moving from start to finish[cite: 1]:
1. **Dataset Preparation:** A key aspect required to set up the fine-tuning process[cite: 1].
2. **Stage 2 Execution:** (Subsequent training steps following data readiness)[cite: 1].
3. **Stage 3 Completion:** (Final steps to complete the three-stage process)[cite: 1].

## 7.2 Preparing a dataset for supervised instruction fine-tuning

### Listing 7.1 Downloading the dataset

In [1]:
import json
import os
import urllib.request # Note: The book explicitly imports urllib.request

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
            
    # Notice this is completely outside the IF block:
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


In [4]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [5]:
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


# Prompt Styles in Instruction Fine-Tuning: Summary

## 1. Notable Examples & Diversity
* **Alpaca:** One of the pioneering LLMs to publicly and transparently detail its complete instruction fine-tuning process[cite: 1].
* **Phi-3:** An LLM developed by Microsoft, which serves as an example to showcase the overall diversity in available prompt styles[cite: 1].

---

## 2. Industry Standard Adoption
* **Alpaca Prompt Style:** This specific format stands out as one of the most popular configurations in the industry[cite: 1].
* **Historical Impact:** Its widespread adoption is largely driven by the fact that it helped fundamentally define the original foundational approach to instruction fine-tuning[cite: 1]. 
* **Chapter Application:** Due to its popularity and historical importance, the Alpaca style is used exclusively throughout the remainder of the chapter[cite: 1].

### Listing 7.2 Implementing the prompt formatting function

In [6]:
def format_input(entry):
    instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [7]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [8]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


### Listing 7.3 Partitioning the dataset

In [9]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion
train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## 7.3 Organizing data into training batches

# Data Batching for Instruction Fine-Tuning: Summary

## 1. Default vs. Custom Collate Functions
* **Default Collate Function:** In typical workflows, the PyTorch `DataLoader` automatically merges individual data samples into training batches using a default collate function[cite: 1].
* **Function Purpose:** A collate function is responsible for consolidating individual data samples into a single batch, ensuring the model can process the data efficiently during training[cite: 1].
* **The Fine-Tuning Shift:** Instruction fine-tuning involves complex data formatting, requiring the creation of a **custom collate function** to be plugged into the `DataLoader`[cite: 1].

---

## 2. Dataset Preparation and Preprocessing Steps
* **Multi-Step Execution:** The batching pipeline breaks down into several progressive steps, including writing the custom collate logic[cite: 1].
* **The `InstructionDataset` Class:** This custom class handles the initial preparation steps (specifically steps 2.1 and 2.2)[cite: 1].
* **Initialization & Tokenization:** Similar to the `SpamDataset` from previous implementations, the `__init__` constructor method of `InstructionDataset` formats the inputs (`format_input`) and pretokenizes all dataset entries upfront[cite: 1].

### Listing 7.4 Implementing an instruction dataset class

In [10]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)

# Batch Acceleration and Padding Strategy: Summary

## 1. Batch Training Acceleration
* **Training Efficiency:** Similar to classification fine-tuning, training is accelerated by collecting multiple training examples into a single batch.
* **Padding Requirement:** Aggregating multiple examples necessitates padding all inputs within a batch to a uniform length.

---

## 2. Padding Token and ID Implementation
* **Token Choice:** As established in classification fine-tuning, the `<|endoftext|>` token serves as the designated padding token.
* **Direct Token ID Appending:** Rather than modifying the raw text inputs, the token ID corresponding to `<|endoftext|>` is appended directly to the pretokenized inputs.
* **Identifying the Token ID:** The tokenizer's `.encode` method can be run on the `<|endoftext|>` token to retrieve the exact token ID needed for padding.

In [11]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


# Custom Collate Function and Dynamic Padding: Summary

## 1. Custom Collate Function Implementation
* **Workflow Progression:** Advancing to step 2.3 of the training pipeline involves adopting a more sophisticated approach by developing a custom collate function.
* **Data Loader Integration:** This newly developed custom function is designed to be passed directly to the PyTorch data loader.

---

## 2. Dynamic Padding Optimization
* **Variable Batch Lengths:** The custom collate function pads the training examples within each individual batch to the same length, while allowing different batches across the dataset to maintain different total lengths.
* **Padding Minimization:** This approach significantly minimizes unnecessary padding tokens.
* **Sequence Extension Strategy:** Instead of padding the entire dataset to a single fixed maximum length, sequences are only extended to match the longest sequence present within that specific batch.

In [12]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
    ):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
            )
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

The custom_collate_draft_1 we implemented is designed to be integrated into a
PyTorch DataLoader, but it can also function as a standalone tool. Here, we use it
independently to test and verify that it operates as intended. Let’s try it on three different inputs that we want to assemble into a batch, where each example gets padded
to the same length:

In [13]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
 inputs_1,
 inputs_2,
 inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


# Batch Creation with Target Token IDs: Summary

## 1. Expanding the Custom Collate Function
* **The Baseline:** The first custom collate function was initially implemented to handle the creation of batches from lists of tokenized inputs.
* **The Expansion:** The custom collate function must be modified to return both the input token IDs and the corresponding target token IDs simultaneously.

---

## 2. Role of Target IDs in Training
* **Representation:** Target token IDs explicitly represent the exact output sequence that the model is expected to generate.
* **Loss Calculation:** These IDs are vital during the training process to compute the loss, which directly drives the model's weight updates.

Similar to the process we used to pretrain an LLM, the target token IDs match the
input token IDs but are shifted one position to the right. This setup, as shown in figure 7.10, allows the LLM to learn how to predict the next token in a sequence.

In [14]:
import torch

def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the maximum length in the batch (plus 1 for the target shift)
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        # Pad the sequence to match the batch_max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        
        # Create inputs (all tokens except the last) and targets (shifted by 1)
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)
        
    # Stack the lists into tensors and move them to the target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    
    return inputs_tensor, targets_tensor

# Execution (Make sure 'batch' is defined before running this)
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


# Target ID Masking and Token Retention: Summary

## 1. Padding Exclusion in Loss Calculation
* **Placeholder Value:** A value of -100 is assigned to all padding tokens within the target list[cite: 1].
* **Loss Calculation Exclusion:** This placeholder value excludes the padding tokens from contributing to the training loss calculation[cite: 1].
* **Learning Impact:** Excluding padding ensures that only meaningful data influences the model's learning process[cite: 1].
* **Difference from Classification:** This step was unnecessary during classification fine-tuning because the model was trained exclusively on the single, final output token[cite: 1].

---

## 2. End-of-Text Token Retention
* **Token Retention:** One `<|endoftext|>` token (ID 50256) is intentionally kept in the target list[cite: 1].
* **Generation Timing:** Retaining this token teaches the LLM precisely when it should generate an end-of-text token in response to an instruction[cite: 1].
* **Completion Indicator:** The generated end-of-text token serves as a clear signal that the model's response is complete[cite: 1].

---

## 3. Function Modifications
* **Collate Logic Update:** The custom collate function is updated to automatically replace target list tokens matching ID 50256 with the -100 value[cite: 1].
* **Maximum Length Parameter:** An `allowed_max_length` parameter is introduced to optionally restrict the length of data samples[cite: 1].
* **Context Size Protection:** This length adjustment is designed to protect model memory when working with custom datasets that exceed the 1,024-token context limit of GPT-2[cite: 1].